In [0]:
# ==========================================
# CELL 1: Storage Authentication
# ==========================================

storage_account = "healthcarestoragerev01"
container = "input"

storage_key = dbutils.secrets.get(
    scope="healthcare-scope",
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

bronze_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/bronze"
quarantine_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/quarantine"

print("Storage authentication configured")
print("Bronze Path:", bronze_path)
print("Quarantine Path:", quarantine_path)

Storage authentication configured
Bronze Path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze
Quarantine Path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/quarantine


In [0]:
# ==========================================
# CELL 1: Configuration
# ==========================================

storage_account = "healthcarestoragerev01"
container = "input"

bronze_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/bronze"
quarantine_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/quarantine"

print("Bronze Path:", bronze_path)
print("Quarantine Path:", quarantine_path)

Bronze Path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze
Quarantine Path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/quarantine


In [0]:
# ==========================================
# CELL 2: Check Bronze Data
# ==========================================

display(dbutils.fs.ls(bronze_path))

path,name,size,modificationTime
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/,departments/,0,1788278227000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/encounters/,encounters/,0,1788278247000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/insurance_claim_data/,insurance_claim_data/,0,1788278253000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/patients/,patients/,0,1788278259000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/providers/,providers/,0,1788278264000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/transactions/,transactions/,0,1788278268000


In [0]:
# ==========================================
# CELL 3: Create Quarantine Folder
# ==========================================

dbutils.fs.mkdirs(quarantine_path)

print("Quarantine folder created successfully")

Quarantine folder created successfully


In [0]:
# ==========================================
# CELL 4: Read Bronze CSV Files
# ==========================================

from pyspark.sql import functions as F

files = [
    "departments.csv",
    "encounters.csv",
    "insurance_claim_data.csv",
    "patients.csv",
    "providers.csv",
    "transactions.csv"
]

print("Bronze files:")
for file in files:
    print(file)

Bronze files:
departments.csv
encounters.csv
insurance_claim_data.csv
patients.csv
providers.csv
transactions.csv


In [0]:
# ==========================================
# CELL 5: Validate Bronze Delta Tables
# Invalid records -> Quarantine
# ==========================================

from pyspark.sql import functions as F

bronze_items = dbutils.fs.ls(bronze_path)

for item in bronze_items:

    name = item.name.rstrip("/")
    input_path = item.path

    # Process only folders
    if not item.isDir():
        continue

    print(f"Processing: {name}")

    # Read Bronze Delta table
    df = (
        spark.read
        .format("delta")
        .load(input_path)
    )

    # Identify completely empty rows
    all_null_condition = F.lit(True)

    for c in df.columns:
        all_null_condition = (
            all_null_condition & F.col(c).isNull()
        )

    invalid_df = df.filter(all_null_condition)
    valid_df = df.filter(~all_null_condition)

    invalid_count = invalid_df.count()
    valid_count = valid_df.count()

    # Quarantine location
    quarantine_file_path = f"{quarantine_path}/{name}"

    # Write invalid records to Quarantine
    if invalid_count > 0:

        (
            invalid_df
            .write
            .format("delta")
            .mode("overwrite")
            .save(quarantine_file_path)
        )

    print(
        f"{name} -> Valid: {valid_count}, "
        f"Invalid: {invalid_count}"
    )

print("Validation completed successfully")

Processing: departments
departments -> Valid: 20, Invalid: 0
Processing: encounters
encounters -> Valid: 10000, Invalid: 0
Processing: insurance_claim_data
insurance_claim_data -> Valid: 10000, Invalid: 0
Processing: patients
patients -> Valid: 5000, Invalid: 0
Processing: providers
providers -> Valid: 25, Invalid: 0
Processing: transactions
transactions -> Valid: 10000, Invalid: 0
Validation completed successfully


In [0]:
# ==========================================
# CELL 6: Verify Quarantine Layer
# ==========================================

display(dbutils.fs.ls(quarantine_path))

[]